In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext line_profiler

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import pickle

import time
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10 * 2.54})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}"
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

In [ ]:
import exciting_environments as excenvs

import dmpe
from dmpe.models import NeuralEulerODEPendulum, NeuralODEPendulum, NeuralEulerODE, NeuralEulerODECartpole
from dmpe.models.model_utils import simulate_ahead_with_env
from dmpe.models.model_training import ModelTrainer
from dmpe.excitation import loss_function, Exciter

from dmpe.utils.density_estimation import (
    update_density_estimate_single_observation, update_density_estimate_multiple_observations, DensityEstimate, select_bandwidth
)
from dmpe.utils.signals import aprbs
from dmpe.evaluation.plotting_utils import (
    plot_sequence, append_predictions_to_sequence_plot, plot_sequence_and_prediction, plot_model_performance
)
from dmpe.evaluation.experiment_utils import (
    get_experiment_ids, load_experiment_results, quick_eval, evaluate_experiment_metrics, evaluate_algorithm_metrics, evaluate_metrics
)

from dmpe.evaluation.experiment_utils import extract_metrics_over_timesteps

---

In [ ]:
from dmpe.evaluation.experiment_utils import get_organized_experiment_ids
from dmpe_params import get_target_distribution
from eval_dmpe import setup_env

full_results_path = "/home/hvater@uni-paderborn.de/projects/forks/DMPE/eval/pmsm/results/dmpe/NODE/"
organized_experiment_ids = get_organized_experiment_ids(full_results_path)
print(organized_experiment_ids[True][2000.0])

In [ ]:
def extract_results(
    lengths,
    raw_results_path,
    algo_names,
    interpolate_to_lengths,
    metrics=None,
    metric_params=None,
    extra_folders=None,
    force_consider_actions=False,
):

    all_results_by_metric = {algo_name: {} for algo_name in algo_names}
    
    for (algo_name, use_interpolation) in zip(algo_names, interpolate_to_lengths):
        full_results_path = raw_results_path / pathlib.Path(algo_name)
        full_results_path = full_results_path / pathlib.Path(extra_folders) if extra_folders is not None else full_results_path

        print("Extract results for", algo_name, "\n at", full_results_path)

        organized_experiment_ids = get_organized_experiment_ids(full_results_path, force_consider_actions=force_consider_actions)

        for ca in organized_experiment_ids.keys():


            print("Momentary experiments consider action distribution:", ca)

            specific_metrics = {}
            for metric_name, metric_function in metrics.items():
                if metric_name == "jsd":
                    target_distribution = get_target_distribution(
                        metric_params[metric_name]["points_per_dim"],
                        metric_params[metric_name]["bandwidth"],
                        metric_params[metric_name]["grid_extend"],
                        ca,
                        metric_params[metric_name]["penalty_function"]
                    )
                    specific_metrics[metric_name] = partial(
                        metric_function,
                        points_per_dim=metric_params[metric_name]["points_per_dim"],
                        bandwidth=metric_params[metric_name]["bandwidth"],
                        target_distribution=target_distribution,
                        ca=ca,
                    )
                elif metric_name == "mcudsa" or metric_name == "kfsc":
                    specific_metrics[metric_name] = partial(
                        metric_function, ca=ca, **metric_params[metric_name]
                    )
                else:
                    specific_metrics[metric_name] = metric_function


            if ca not in all_results_by_metric[algo_name]:
                all_results_by_metric[algo_name][ca] = {}

            for rpm in organized_experiment_ids[ca].keys():
                print(f"Momentary experiments run at {int(rpm)} rpm.")
            
                if not use_interpolation:
                    all_results_by_metric[algo_name][ca][rpm] = extract_metrics_over_timesteps(
                        experiment_ids=organized_experiment_ids[ca][rpm],
                        results_path=full_results_path,
                        lengths=lengths,
                        metrics=specific_metrics,
                    )
                else:
                    all_results_by_metric[algo_name][ca][rpm] = extract_metrics_over_timesteps_via_interpolation(
                        experiment_ids=organized_experiment_ids[ca][rpm],
                        results_path=full_results_path,
                        target_lengths=lengths,
                        metrics=specific_metrics,
                    )
                print("\n")
    return all_results_by_metric

In [ ]:
from eval_dmpe import setup_env
_, penalty_function = setup_env(0)

lengths = jnp.linspace(0, 15000, 151, dtype=jnp.int32)
lengths

# Extract:

## DMPE:

In [ ]:
system_name = "pmsm"

dmpe_pmsm_results_by_metric = extract_results(
    lengths=lengths,
    raw_results_path=pathlib.Path("/home/hvater@uni-paderborn.de/projects/forks/DMPE/eval/pmsm/results/dmpe/"),
    algo_names=["NODE", "PM", "RLS"],
    interpolate_to_lengths=[False, False, False],
    extra_folders=None,
    metrics={
        "sc": lambda observations, actions: penalty_function(observations, actions) / (observations.shape[0] * 1e3), 
        # lambda observations, actions: penalty_function(observations, None),
    },
)


In [ ]:
with open("results/quantitative_metrics_per_time/dmpe_pmsm_constraint_violations.pickle", "wb") as handle:
    pickle.dump(dmpe_pmsm_results_by_metric, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
dmpe_pmsm_results_by_metric.keys()

## iGOATS:

In [ ]:
system_name = "pmsm"

igoats_pmsm_results_by_metric = extract_results(
    lengths=lengths,
    raw_results_path=pathlib.Path("/home/hvater@uni-paderborn.de/projects/forks/DMPE/eval/pmsm/results/"),
    algo_names=["igoats"],
    interpolate_to_lengths=[False],
    extra_folders=None,
    metrics={
        "sc": lambda observations, actions: penalty_function(observations, actions) / (observations.shape[0] * 1e3), 
        # lambda observations, actions: penalty_function(observations, None),
    },
)

In [ ]:
with open("results/quantitative_metrics_per_time/igoats_pmsm_constraint_violations.pickle", "wb") as handle:
    pickle.dump(igoats_pmsm_results_by_metric, handle, protocol=pickle.HIGHEST_PROTOCOL)

## Heuristics:

In [ ]:
heuristics_pmsm_results_by_metric = extract_results(
    lengths=lengths,
    raw_results_path=pathlib.Path("/home/hvater@uni-paderborn.de/projects/forks/DMPE/eval/pmsm/results/heuristics"),
    algo_names=["random_walk", "current_plane_sweep"],
    interpolate_to_lengths=[False, False],
    extra_folders=None,
    metrics={
        "sc": lambda observations, actions: penalty_function(observations, actions) / (observations.shape[0] * 1e3), 
        # lambda observations, actions: penalty_function(observations, None),
    },
    force_consider_actions=True
)

In [ ]:
with open("results/quantitative_metrics_per_time/heuristics_pmsm_constraint_violations.pickle", "wb") as handle:
    pickle.dump(heuristics_pmsm_results_by_metric, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Plot:

In [ ]:
from copy import deepcopy

In [ ]:
full_column_width = 18.2
half_column_width = 8.89

def plot_metrics_by_sequence_length_for_all_algos_for_all_rpm(data_per_algo_with_rpm, lengths, algo_names, use_log=False, plot_log=False):
    assert len(data_per_algo_with_rpm) == len(algo_names), "Mismatch in number of algo results and number of algo names"

    rpms = list(data_per_algo_with_rpm[0].keys())
    
    metric_keys = list(data_per_algo_with_rpm[0][rpms[0]].keys())
    #metric_keys.remove("ae")
    
    fig, axs = plt.subplots(1, len(rpms), figsize=(full_column_width, 4), sharex=True, sharey="row") # figsize=(19, 18)

    for rpm_idx, rpm in enumerate(rpms):

        data_per_algo = [element[rpm] for element in data_per_algo_with_rpm]
        colors = plt.rcParams["axes.prop_cycle"]()

        for algo_name, data in zip(algo_names, data_per_algo):
            
            c = next(colors)["color"]
            if c == '#d62728':
                c = next(colors)["color"]
            for metric_idx, metric_key in enumerate(metric_keys):
                
                mean = jnp.nanmean(jnp.log(data[metric_key]), axis=0) if use_log else jnp.nanmean(data[metric_key], axis=0)
                std = jnp.nanstd(jnp.log(data[metric_key]), axis=0) if use_log else jnp.nanstd(data[metric_key], axis=0)


                if algo_name=="$\mathrm{RLS-DMPE}$":
                    style = "dashed"
                elif algo_name=="$\mathrm{iGOATS}$":
                    style = "dashdot"
                elif algo_name=="$\mathrm{PM-DMPE}$":
                    style = "dotted"
                else:
                    style=None

                
                axs[rpm_idx].plot(
                    lengths * 1e-4,
                    mean,  # jnp.log(mean) if use_log else mean,
                    label=algo_name if rpm_idx == 0 and metric_idx == 0 else None,
                    color=c,
                    linewidth=2.5,
                    linestyle=style,
                )
                # axs[rpm_idx].fill_between(
                #     lengths * 1e-4,
                #     mean - std,  # jnp.log(mean - std) if use_log else mean - std,
                #     mean + std,  # jnp.log(mean + std) if use_log else mean + std,
                #     color=c,
                #     alpha=0.1,
                # )
            # axs[metric_idx].set_ylabel(("log " if use_log else "") + metric_key)

    if plot_log:
        for ax in axs:
                ax.set_yscale('log', base=10)
    
        #axs[-1].set_yscale('log', base=10) if list(metric_keys)[-1] == "ksfc" else None

    for idx, metric_key in enumerate(metric_keys):
        axs[0].set_ylabel(f"$\mathcal{{L}}_\mathrm{{{metric_key.upper()}}}$")

    for ax in axs:
        ax.set_xlabel("$t$ $\mathrm{in}$ $\mathrm{s}$")
        ax.set_xlim(lengths[0] * 1e-4 - 0.02, lengths[-1] * 1e-4 + 0.02)
        ax.set_xticks((0, 0.5, 1.0, 1.5))

        xtick_labels = ax.get_xticklabels()
        xtick_labels[0].set_ha('left')
        xtick_labels[-1].set_ha('right')

    for ax in axs:
        ax.grid(True, which="both", alpha=0.3)
        ax.tick_params(axis="y", direction='in')
        ax.tick_params(axis="x", direction='in') 
    fig.legend(prop={'size': 8 * 2.54}, framealpha=0.5, loc="center", bbox_to_anchor=(0.525, 0.0), fancybox=True, shadow=False,  ncol=len(algo_names))
    # axs[0].legend(prop={'size': 7 * 2.54}, framealpha=0.5, loc="upper left")

    for ax, col in zip(axs, ["$0$ $\mathrm{min}^{-1}$", "$3000$ $\mathrm{min}^{-1}$", "$5000$ $\mathrm{min}^{-1}$", "$7000$ $\mathrm{min}^{-1}$", "$9000$ $\mathrm{min}^{-1}$",]):
        ax.set_title(col)

    # plt.subplots_adjust(hspace=0.1, wspace=0.0)
    
    plt.tight_layout(w_pad=0.4)
    fig.align_ylabels(axs)

    return fig

In [ ]:
lengths = jnp.linspace(0, 15000, 151, dtype=jnp.int32)
lengths

# load:

with open("results/quantitative_metrics_per_time/dmpe_pmsm_constraint_violations.pickle", 'rb') as handle:
    dmpe_pmsm_results_by_metric = pickle.load(handle)

with open("results/quantitative_metrics_per_time/heuristics_pmsm_constraint_violations.pickle", 'rb') as handle:
    heuristics_pmsm_results_by_metric = pickle.load(handle)


with open("results/quantitative_metrics_per_time/igoats_pmsm_constraint_violations.pickle", 'rb') as handle:
    igoats_pmsm_results_by_metric = pickle.load(handle)

In [ ]:
current_plane_sweep_results = heuristics_pmsm_results_by_metric["current_plane_sweep"][True]
random_walk_results = heuristics_pmsm_results_by_metric["random_walk"][True]

NODE_results = deepcopy(dmpe_pmsm_results_by_metric["NODE"][True])
RLS_results = dmpe_pmsm_results_by_metric["RLS"][True]
PM_results = dmpe_pmsm_results_by_metric["PM"][True]

iGOATS_results = igoats_pmsm_results_by_metric["igoats"][True]


for rpm in NODE_results.keys():
    # exclude crashed runs:    
    if rpm == 7000:
        for key in NODE_results[rpm].keys():
            NODE_results[rpm][key] = jnp.concatenate([NODE_results[rpm][key][:5],  NODE_results[rpm][key][7:]], axis=0)
    elif rpm == 9000:
        for key in NODE_results[rpm].keys():
            NODE_results[rpm][key] =  jnp.concatenate([NODE_results[rpm][key][:2], NODE_results[rpm][key][3:5], NODE_results[rpm][key][7:]], axis=0)

    print(len(NODE_results[rpm]["sc"]))

plot_metrics_by_sequence_length_for_all_algos_for_all_rpm(        
    data_per_algo_with_rpm=[current_plane_sweep_results, random_walk_results, NODE_results, RLS_results, iGOATS_results, PM_results],
    lengths=lengths,
    algo_names=["$\mathrm{PI-sweep}$", "$\mathrm{random-walk}$", "$\mathrm{NODE-DMPE}$", "$\mathrm{RLS-DMPE}$", "$\mathrm{iGOATS}$",  "$\mathrm{PM-DMPE}$"],
    use_log=False,
    plot_log=True,
);

plt.savefig("results/quantitative_metrics_per_time/all_constraint_violations_all_rpm_all_algos.pdf", bbox_inches='tight')